# Questão 6
## 1. Motivação
Como se compara a composição da Câmara dos Deputados com a população brasileira?

As questões anteriores mostraram que a proporção de mulheres entre as candidaturas aumentou depois da obrigatoriedade da cota, mas que esse crescimento não se traduziu na mesma proporção de mulheres eleitas. A comparação entre candidatas e eleitas (questão 2) permite avaliar a eleição; aqui, damos mais um passo e comparamos a representação política com a distribuição de mulheres e homens na população.

Para que a comparação use períodos compatíveis, consideramos as pessoas eleitas para deputado federal na eleição ordinária de 2022 e a população registrada no Censo Demográfico de 2022. Como a população não é o mesmo conjunto que o eleitorado, a comparação não mede diretamente o comportamento do voto: ela mede a distância entre a composição por sexo da população e a composição, por gênero registrado, dos representantes eleitos.

Traduzimos a pergunta em duas perguntas de dados:

1. **Brasil:** qual é a diferença entre a fração de mulheres na população e a fração de mulheres entre os deputados federais eleitos em 2022?
2. **Estados:** essa diferença aparece em todas as UFs? Quão diferentes são as bancadas de deputados federais de cada estado em relação à população do próprio estado?

## 2. Hipótese
Esperamos que as mulheres sejam pouco mais da metade da população brasileira, mas uma parcela bem menor dos deputados federais eleitos. Nas UFs, esperamos que a participação feminina nas bancadas também fique abaixo da participação feminina entre os residentes, embora a intensidade da diferença varie entre estados por causa do número reduzido de vagas de algumas bancadas.

A hipótese nacional seria refutada se a fração de deputadas eleitas fosse próxima ou superior à fração de mulheres na população. Nos estados, a hipótese seria refutada onde a fração de eleitas fosse igual ou superior à fração de mulheres na população da UF.

## 3. Dados

### 3.1 Deputados federais eleitos em 2022

Usamos `resultados_candidato.parquet`, a mesma base de resultados do TSE via Base dos Dados utilizada nas questões 2 a 5. Selecionamos eleições ordinárias de 2022, o cargo de deputado federal e os registros cujo resultado começa por `eleito` (incluindo eleitos por quociente partidário e por média, sem contar suplentes). Mantemos somente os gêneros `feminino` e `masculino`, usados nas análises anteriores.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests

pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"
COR_GENERO = {"Feminino": "#B5338A", "Masculino": "#2B7BB9"}   # mesmas cores das questões 4 e 5

ARQUIVO_TSE = Path("../data/resultados_candidato.parquet")
if not ARQUIVO_TSE.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO_TSE}. Rode primeiro o notebook de download do projeto."
    )

COLUNAS = ["ano", "tipo_eleicao", "cargo", "genero", "resultado", "sigla_uf"]
df = pd.read_parquet(ARQUIVO_TSE, columns=COLUNAS)

base = df[
    (df["ano"] == 2022)
    & (df["tipo_eleicao"] == "eleicao ordinaria")
    & (df["cargo"] == "deputado federal")
    & (df["genero"].isin(["feminino", "masculino"]))
].copy()
base["eleito"] = base["resultado"].fillna("").str.startswith("eleito")
eleitos = base[base["eleito"]].copy()
eleitos["Gênero"] = eleitos["genero"].map({"feminino": "Feminino", "masculino": "Masculino"})

print("Candidaturas a deputado federal em 2022:", len(base))
print("Eleitos identificados:", len(eleitos))
print(eleitos["Gênero"].value_counts(dropna=False))

In [ ]:
# Checagem da cobertura: a Câmara tem 513 cadeiras, e o resultado oficial de 2022 foi de 91 mulheres.
# Se os valores não coincidirem, precisamos revisar o filtro ou a versão dos dados antes de interpretar.
n_eleitos = len(eleitos)
n_mulheres = int((eleitos["genero"] == "feminino").sum())
n_homens = int((eleitos["genero"] == "masculino").sum())

assert n_eleitos == 513, f"Esperados 513 eleitos; encontrados {n_eleitos}."
assert n_mulheres == 91, f"Esperadas 91 mulheres eleitas; encontradas {n_mulheres}."
assert n_mulheres + n_homens == n_eleitos
assert eleitos["sigla_uf"].nunique() == 27, "Nem todas as 27 UFs aparecem nos resultados."

print("Conferência com Câmara dos Deputados (eleição de 2022): OK")

### 3.2 População brasileira em 2022

Usamos os totais oficiais do Censo Demográfico 2022 do IBGE: 104.548.325 mulheres e 98.532.431 homens. Esses valores são contagens de residentes, não de eleitores, e permitem calcular as proporções nacionais a partir dos números originais, sem arredondar antes da análise.


In [ ]:
pop_br = pd.DataFrame({
    "Gênero": ["Feminino", "Masculino"],
    "populacao": [104_548_325, 98_532_431],
})
pop_br["pct_pop"] = pop_br["populacao"] / pop_br["populacao"].sum() * 100

print("População total no Censo 2022:", f"{pop_br['populacao'].sum():,}".replace(",", "."))
pop_br.round({"pct_pop": 2})

### 3.3 População por UF (IBGE/SIDRA)

Para comparar as bancadas estaduais, consultamos a tabela 9514 do SIDRA/IBGE, variável 93 (*população residente*), ano de 2022, sexo feminino/masculino, idade total e forma de declaração total. O primeiro acesso à API salva uma cópia em `data/populacao_uf_sexo_2022.csv`, para não ser necessário refazer a consulta em todas as execuções.


In [ ]:
UF_POR_CODIGO = {
    11: "RO", 12: "AC", 13: "AM", 14: "RR", 15: "PA", 16: "AP", 17: "TO",
    21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE",
    27: "AL", 28: "SE", 29: "BA", 31: "MG", 32: "ES", 33: "RJ",
    35: "SP", 41: "PR", 42: "SC", 43: "RS", 50: "MS", 51: "MT",
    52: "GO", 53: "DF",
}
ARQUIVO_IBGE_UF = Path("../data/populacao_uf_sexo_2022.csv")
URL_SIDRA = (
    "https://apisidra.ibge.gov.br/values/t/9514/n3/all/v/93/p/2022"
    "/c2/4,5/c287/100362/c286/113635"
)

def baixar_populacao_uf():
    resposta = requests.get(URL_SIDRA, headers={"Accept": "application/json"}, timeout=75)
    resposta.raise_for_status()
    linhas = resposta.json()
    if len(linhas) <= 1:
        raise ValueError("O SIDRA não retornou dados por UF.")
    # Na API SIDRA, a primeira linha descreve as colunas (D1C, D4N, V etc.).
    dados = pd.DataFrame(linhas[1:]).rename(columns=linhas[0])
    coluna_codigo = next(c for c in dados if "Unidade da Federação (Código)" in c)
    coluna_sexo = next(c for c in dados if c.strip() == "Sexo")
    coluna_valor = next(c for c in dados if c.strip() == "Valor")
    saida = pd.DataFrame({
        "sigla_uf": pd.to_numeric(dados[coluna_codigo], errors="raise").map(UF_POR_CODIGO),
        "Gênero": dados[coluna_sexo].replace({"Mulheres": "Feminino", "Homens": "Masculino"}),
        "populacao": pd.to_numeric(dados[coluna_valor], errors="raise").astype("int64"),
    })
    return saida

if ARQUIVO_IBGE_UF.exists():
    pop_uf = pd.read_csv(ARQUIVO_IBGE_UF)
    print("IBGE por UF: carregado do arquivo local.")
else:
    try:
        pop_uf = baixar_populacao_uf()
        # Só salva se a resposta for coerente com os dados publicados do IBGE.
        assert len(pop_uf) == 54 and pop_uf["sigla_uf"].nunique() == 27
        assert pop_uf["populacao"].sum() == pop_br["populacao"].sum()
        ARQUIVO_IBGE_UF.parent.mkdir(exist_ok=True, parents=True)
        pop_uf.to_csv(ARQUIVO_IBGE_UF, index=False)
        print("IBGE por UF: baixado e salvo em", ARQUIVO_IBGE_UF)
    except (requests.RequestException, ValueError, AssertionError, StopIteration, KeyError) as erro:
        # A comparação nacional continua disponível mesmo se a API estiver fora do ar.
        pop_uf = pd.DataFrame(columns=["sigla_uf", "Gênero", "populacao"])
        print("Não foi possível consultar o SIDRA:", erro)
        print("A comparação por estado (seção 5) requer repetir esta célula com internet.")

if not pop_uf.empty:
    assert len(pop_uf) == 54 and pop_uf["sigla_uf"].nunique() == 27
    assert pop_uf["populacao"].sum() == pop_br["populacao"].sum()
    display(pop_uf.head())

### 3.4 Decisões

- **Recorte temporal:** escolhemos 2022 para as duas fontes. Contamos pessoas eleitas em 2022, não parlamentares que estão em exercício atualmente, pois a composição pode mudar com licenças, suplências e trocas durante a legislatura.
- **Definição de eleitos:** usamos a mesma regra das questões 2 a 5: resultado começando por `eleito`, sem contar suplentes. Conferimos o total de 513 cadeiras e as 91 deputadas eleitas divulgado pela Câmara dos Deputados.
- **Definição de gênero:** a base eleitoral usa `feminino`/`masculino`, enquanto o IBGE apresenta sexo. Usamos as categorias publicadas, mas elas não são conceitualmente idênticas e não capturam adequadamente toda a diversidade de gênero.
- **Universo:** o Censo inclui todos os residentes, inclusive quem não pode votar. Por isso, a comparação não é uma taxa de sucesso eleitoral nem uma medida de preferências dos eleitores.
- **Estados:** comparamos cada bancada à população residente da UF pela qual os deputados foram eleitos. Bancadas pequenas têm proporções mais instáveis: em uma bancada com oito vagas, uma deputada a mais altera o percentual em 12,5 pontos.

## 4. Brasil: população e deputados federais eleitos, 2022

Calculamos a proporção de mulheres e homens nos dois grupos e a diferença, em pontos percentuais, entre a participação feminina na Câmara e na população. Também calculamos a razão entre as duas proporções: valor 1 significaria participações iguais; valor menor que 1 indica sub-representação feminina na Câmara em relação à população.

In [ ]:
camara_br = (
    eleitos.groupby("Gênero", observed=True).size()
    .rename("eleitos").reset_index()
)
camara_br["pct_camara"] = camara_br["eleitos"] / camara_br["eleitos"].sum() * 100
comp_br = pop_br.merge(camara_br, on="Gênero", validate="one_to_one")
comp_br["dif_pp"] = comp_br["pct_camara"] - comp_br["pct_pop"]

fem = comp_br.loc[comp_br["Gênero"] == "Feminino"].iloc[0]
pct_pop_f = float(fem["pct_pop"])
pct_camara_f = float(fem["pct_camara"])
dif_pp_br = float(fem["dif_pp"])
razao_rep = pct_camara_f / pct_pop_f

print(f"Mulheres na população: {pct_pop_f:.2f}%")
print(f"Mulheres entre eleitos: {pct_camara_f:.2f}%")
print(f"Diferença (Câmara − população): {dif_pp_br:+.2f} pontos percentuais")
print(f"Razão entre as participações: {razao_rep:.2f}")
comp_br.round(2)

In [ ]:
# Figura 1: composição percentual completa, sem eixo truncado.
long_br = comp_br.melt(
    id_vars=["Gênero", "populacao", "eleitos"],
    value_vars=["pct_pop", "pct_camara"],
    var_name="grupo", value_name="pct",
)
long_br["grupo"] = long_br["grupo"].map({
    "pct_pop": "População brasileira",
    "pct_camara": "Eleitos para a Câmara",
})
long_br["pct"] = long_br["pct"].astype(float)

fig1 = px.bar(
    long_br, x="grupo", y="pct", color="Gênero", text="pct",
    barmode="stack", color_discrete_map=COR_GENERO,
    category_orders={"Gênero": ["Feminino", "Masculino"],
                     "grupo": ["População brasileira", "Eleitos para a Câmara"]},
    labels={"grupo": "", "pct": "Participação (%)", "Gênero": ""},
    title="Em 2022, mulheres eram maioria da população, mas 17,7% dos deputados federais eleitos",
)
fig1.update_traces(texttemplate="%{text:.1f}%", textposition="inside", textfont_size=13)
fig1.update_yaxes(range=[0, 100], ticksuffix="%", dtick=20)
fig1.update_layout(height=470, legend_title_text="", uniformtext_minsize=11,
                   margin=dict(t=90, b=50, l=40, r=30))
fig1.show()

### Análise: Brasil

A Figura 1 mostra uma diferença grande entre a composição da população e a dos deputados federais eleitos em 2022. Segundo o IBGE, 51,48% dos residentes eram mulheres (104.548.325 de 203.080.756), contra 48,52% de homens. Entre os eleitos para as 513 cadeiras da Câmara, foram 91 mulheres (17,74%) e 422 homens (82,26%). Assim, a participação feminina entre os eleitos ficou 33,74 pontos percentuais abaixo de sua participação na população. A razão entre as participações é de aproximadamente 0,34: a proporção de mulheres na Câmara corresponde a cerca de um terço de sua proporção na população.

O resultado é compatível com o que vimos nas questões 1 e 2. A regra de cotas aumentou a presença de candidatas, mas as mulheres ainda representam uma parcela bem menor dos eleitos e, consequentemente, da Câmara. A diferença observada, sozinha, **não prova** que a causa seja a regra de cotas, o financiamento, os partidos ou o comportamento dos eleitores; trata-se de uma comparação descritiva.

## 5. Estados: diferença entre a população e a bancada federal, 2022

Para cada UF, calculamos (1) a fração de mulheres residentes; (2) a fração de mulheres entre os deputados federais eleitos; e (3) a diferença entre elas em pontos percentuais. Uma diferença negativa significa que a participação feminina entre os eleitos é menor que na população. A comparação usa o percentual da bancada de cada UF, não a participação da UF no total de 513 cadeiras.

In [ ]:
if pop_uf.empty:
    comp_uf = pd.DataFrame()
    print("Comparação estadual indisponível: execute a seção 3.3 com acesso ao SIDRA.")
else:
    pop_estado = (
        pop_uf.pivot(index="sigla_uf", columns="Gênero", values="populacao")
        .rename(columns={"Feminino": "mulheres_pop", "Masculino": "homens_pop"})
        .reset_index()
    )
    pop_estado["total_pop"] = pop_estado["mulheres_pop"] + pop_estado["homens_pop"]
    pop_estado["pct_pop_f"] = pop_estado["mulheres_pop"] / pop_estado["total_pop"] * 100

    eleit_estado = (
        eleitos.groupby(["sigla_uf", "Gênero"], observed=True).size()
        .unstack("Gênero", fill_value=0).reset_index()
        .rename(columns={"Feminino": "deputadas", "Masculino": "deputados"})
    )
    eleit_estado["total_eleitos"] = eleit_estado["deputadas"] + eleit_estado["deputados"]
    eleit_estado["pct_eleitas_f"] = eleit_estado["deputadas"] / eleit_estado["total_eleitos"] * 100

    comp_uf = pop_estado.merge(eleit_estado, on="sigla_uf", validate="one_to_one")
    comp_uf["dif_pp"] = comp_uf["pct_eleitas_f"] - comp_uf["pct_pop_f"]
    comp_uf = comp_uf.sort_values("dif_pp")

    assert len(comp_uf) == 27
    assert comp_uf["deputadas"].sum() == 91
    assert comp_uf["total_eleitos"].sum() == 513
    print(f"UFs com diferença negativa: {(comp_uf['dif_pp'] < 0).sum()} de 27")
    display(comp_uf[["sigla_uf", "total_eleitos", "deputadas",
                     "pct_pop_f", "pct_eleitas_f", "dif_pp"]].round(1))

In [ ]:
if not comp_uf.empty:
    # Figura 2: distância em pontos percentuais por UF, com referência nacional.
    figura_uf = comp_uf.copy()
    figura_uf["sigla_uf"] = figura_uf["sigla_uf"].astype(str)
    fig2 = px.bar(
        figura_uf, x="dif_pp", y="sigla_uf", orientation="h",
        color_discrete_sequence=["#B5338A"],
        hover_data={"deputadas": True, "total_eleitos": True,
                    "pct_pop_f": ":.1f", "pct_eleitas_f": ":.1f", "dif_pp": ":.1f"},
        labels={"dif_pp": "Câmara − população (pontos percentuais)",
                "sigla_uf": "UF", "deputadas": "Deputadas eleitas",
                "total_eleitos": "Vagas", "pct_pop_f": "% mulheres na população",
                "pct_eleitas_f": "% mulheres eleitas"},
        title="Em 2022, a presença de mulheres nas bancadas ficou abaixo da população na maioria das UFs",
    )
    fig2.add_vline(x=dif_pp_br, line_color="#555555", line_dash="dash",
                   annotation_text="Brasil", annotation_position="top right")
    fig2.add_vline(x=0, line_color="#333333", line_width=1)
    fig2.update_yaxes(categoryorder="array", categoryarray=figura_uf["sigla_uf"].tolist())
    fig2.update_layout(height=825, showlegend=False,
                       margin=dict(t=95, b=60, l=35, r=40))
    fig2.show()

    melhor = comp_uf.loc[comp_uf["dif_pp"].idxmax()]
    pior = comp_uf.loc[comp_uf["dif_pp"].idxmin()]
    print(f"Maior diferença (Câmara − população): {melhor['sigla_uf']} ({melhor['dif_pp']:+.1f} pp)")
    print(f"Menor diferença (Câmara − população): {pior['sigla_uf']} ({pior['dif_pp']:+.1f} pp)")
else:
    print("Figura 2 não gerada: os dados de UF do SIDRA não estão disponíveis.")

### Análise: estados

A Figura 2 permite verificar se a distância nacional também ocorre nas bancadas de cada UF. Como a proporção de mulheres na população muda relativamente pouco entre estados, as diferenças entre as barras refletem principalmente a variação no número de deputadas eleitas. A tabela e o gráfico identificam as UFs com maior e menor distância, e o código informa quantas ficaram abaixo da proporção feminina da população.

É importante não interpretar diferenças grandes entre estados como prova de que uma UF necessariamente oferece piores condições de participação política. As menores bancadas têm apenas oito cadeiras; uma única deputada representa 12,5% da bancada, o que torna os percentuais sensíveis a pequenas mudanças. Além disso, os dados representam o resultado de uma eleição, não um padrão causal permanente.

## 6. Conclusão

A hipótese nacional foi confirmada: as mulheres eram maioria da população brasileira em 2022 (51,5%), mas ocuparam apenas 17,7% das cadeiras conquistadas na eleição daquele ano para a Câmara dos Deputados. A diferença de 33,7 pontos percentuais mostra que o conjunto de eleitos está distante da composição por sexo da população. Em termos relativos, a participação feminina entre os eleitos equivale a cerca de um terço da participação feminina entre os residentes.

Essa conclusão complementa as questões anteriores. A questão 1 mostrou que a presença de candidatas aumentou depois do preenchimento obrigatório da cota; a questão 2 mostrou que o aumento das candidaturas não se converte na mesma proporção de eleitas; e esta questão evidencia o resultado final dessa trajetória quando comparado à população. Mesmo quando as mulheres constituem a maioria dos residentes, elas seguem minoritárias na representação federal.

A análise por UF permite verificar a variação territorial dessa distância, sem confundir percentuais de bancadas pequenas com tendências estruturais. Os dados indicam uma sub-representação descritiva, mas não permitem identificar, isoladamente, qual mecanismo explica a diferença.

## 7. Limitações

- **População não é eleitorado:** o Censo inclui crianças, pessoas sem direitos políticos e residentes que não votaram; por isso, o parâmetro de 51,5% é demográfico, não uma estimativa de como os votos deveriam ter sido distribuídos.
- **Sexo e gênero:** o IBGE divulga a população por sexo, enquanto os registros eleitorais usam categorias de gênero. A comparação aproxima duas classificações distintas e não cobre toda a diversidade de identidades de gênero.
- **Fotografia de 2022:** a análise usa quem foi eleito em 2022, e não a composição atual da Câmara, que pode mudar com licenças e suplências.
- **Bancadas pequenas:** percentuais por UF são instáveis quando há poucas vagas; a análise estadual é descritiva e não testa causas.
- **Outros aspectos da representação:** não analisamos idade, raça, escolaridade, renda nem ocupação, que também podem revelar diferenças entre representantes e representados.
- **Cobertura dos dados:** se o arquivo local do TSE não reproduzir as 513 cadeiras e as 91 eleitas, o código interrompe a análise para evitar uma conclusão baseada em cobertura incompleta. A parte por UF requer a tabela SIDRA 9514.